# Pitch:timing ratio and rho sweep — Coco-E

Complete `pitch.ipynb` and `note.ipynb` first. This stage uses **only the same 195 CocoChorales source stems**, never Bach10/URMP. It reads the completed dataset-balanced pitch and note winners from `pitch/best.json` and `note/best.json`, validating their artifacts and provenance. It uses the selected note method as well as its four parameters.

Retain the existing E2 injector and instrument-preserving renderer: one clean control and two 25%-mistake performances (seeds 0 and 1), five equally weighted edit types. Thus 195 sources produce 585 performances. Clean-audio pitch/note artifacts cannot substitute for the newly performed audio: infer the selected frontend on each rendered performance, once, and cache the initial notes before testing alignment weights.

Every weight candidate starts from an independent copy of those initial notes and runs production alignment/refit/repeat recovery. Rank pooled **Mistake Event F1** across injected Coco performances, using the existing 100 ms / 50 cents event scorer. Substitutions count as missed + extra. Report clean false positives separately, excluded from F1. No three-dataset weighting applies here because this stage is Coco-only.

Changing either upstream winner selects a fresh frontend/report namespace; unchanged reruns reuse checkpoints. Earlier fixed-default alignment results remain historical and cannot be ranked alongside changed-frontend results.

## Experimental gap model

For an unmatched note of duration d, use **C_gap = alpha × rho + beta × d**. Here alpha is the pitch weight, beta is the timing weight, and rho is a shared insertion/deletion penalty in semitones. This is implemented via per-candidate `ins_cost = del_cost = alpha*rho`; production defaults are unchanged. The same costs feed repeat-recovery deletion fees.

Reduce the previous pitch/time grid to distinct ratios, represented by coprime integer weights (6:2 becomes 3:1). Exclude 0:0. Include pitch-only 1:0 and timing-only 0:1 controls; rho has no effect at alpha=0, so evaluate that endpoint only once. Keep production's duration-only repeat recovery unchanged; normalization is explicitly fixed for reproducibility, including numerical tie tolerances.

Sweep rho = **0.25, 0.5, 1, 1.25, 2, 3, 5 semitones**. The 4:1 / rho=1.25 row reproduces the original 4:1 / gap-intercept=5 cost model. Rank by pooled injected-event F1, with clean false positives reported separately.

Reuse the exact completed Coco-E initial detection + merging caches from the previous linked sweep. Every candidate independently performs alignment, timing refit and repeat splitting. No pitch inference, segmentation or rendering is needed. Results go into a separate `mistake_ratio_rho/<contract-hash>/` directory. This notebook is prepared for manual execution; no full sweep has been launched.


In [1]:
from pathlib import Path
import os, sys
# Limit native threads before importing numerical libraries: 8 processes × 1 thread.
for name in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS',
             'VECLIB_MAXIMUM_THREADS', 'NUMEXPR_NUM_THREADS', 'NUMBA_NUM_THREADS'):
    os.environ[name] = '1'
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'app.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from benchmarks.modules.sweeps import AppendixSweeps as sweep
import pandas as pd
WORKERS = 8
OUTPUT = ROOT / 'benchmarks/results/sweeps/balanced_212_v3'
DATASET_ROOT = ROOT / 'benchmarks/datasets/cocochorales_tiny'
SPLIT = 'test'  # Locally available partition; NEW development cohort, NOT a test claim.
SEED = 20261116

from benchmarks.notebooks.sweeps.scripts import mistake_ratio_sweep as ratio_sweep
import importlib
ratio_sweep = importlib.reload(ratio_sweep)


## Coco-only development cohort

The shared manifest includes all 212 pitch/note tracks. This notebook explicitly filters it to the original 195 Coco stems (15 per instrument). The real datasets affect only the upstream setting selection. These are tuning results, not held-out evaluation.


In [2]:
# Freeze the sample and validate source files; no detector is run here.
manifest = sweep.materialize(OUTPUT, root=DATASET_ROOT, split=SPLIT,
                             seed=SEED, workers=WORKERS)
cohort = pd.DataFrame(manifest['items'])
assert cohort.groupby('dataset').size().to_dict() == {'bach10-original': 4, 'coco': 195, 'urmp': 13}
display(pd.crosstab(cohort.instrument, cohort.dataset))
print('Frozen manifest:', OUTPUT / 'cohort.json')
coco = cohort[cohort.dataset.eq('coco')]
assert len(coco) == 195 and coco.groupby('instrument').size().eq(15).all()
print('Mistake sources:', len(coco), '(Coco only)')


dataset,bach10-original,coco,urmp
instrument,,,
bassoon,1,15,1
cello,0,15,1
clarinet,1,15,1
double bass,0,15,1
flute,0,15,1
horn,0,15,1
oboe,0,15,1
saxophone,1,15,1
trombone,0,15,1


Frozen manifest: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/sweeps/balanced_212_v3/cohort.json
Mistake sources: 195 (Coco only)


## Distinct ratios and semitone gap penalties


In [3]:
RATIOS = ratio_sweep.unique_ratios()
RHO_VALUES = ratio_sweep.RHO_VALUES
INJECTION_SEEDS = (0, 1)
CANDIDATES = ratio_sweep.variants(RATIOS, RHO_VALUES)
print('Distinct pitch:timing ratios:', RATIOS)
print('Candidate count:', len(CANDIDATES), '(rho duplicates removed for alpha=0)')
display(ratio_sweep.parameter_table(CANDIDATES))


Distinct pitch:timing ratios: [(0, 1), (1, 4), (1, 2), (3, 4), (1, 1), (3, 2), (2, 1), (3, 1), (4, 1), (6, 1), (8, 1), (12, 1), (16, 1), (1, 0)]
Candidate count: 92 (rho duplicates removed for alpha=0)


,Pitch:Timing,Rho (semitones),Pitch Weight,Timing Weight,Gap intercept (alpha*rho)
0,0:1,0.00,0.0,1.0,0.00
1,1:4,0.25,1.0,4.0,0.25
2,1:4,0.50,1.0,4.0,0.50
3,1:4,1.00,1.0,4.0,1.00
4,1:4,1.25,1.0,4.0,1.25
...,...,...,...,...,...
87,1:0,1.00,1.0,0.0,1.00
88,1:0,1.25,1.0,0.0,1.25
89,1:0,2.00,1.0,0.0,2.00
90,1:0,3.00,1.0,0.0,3.00


## Run and rank

In [4]:
# Run manually: reuses completed merged-note frontends; recomputes alignment/refinement only.
results = ratio_sweep.run(OUTPUT, ratios=RATIOS, rhos=RHO_VALUES,
                          workers=WORKERS, seeds=INJECTION_SEEDS)
display(results['summary'].head(20))
print('Reports:', results['directory'])
print('Winner:', results['best'])


17940/17940 | test__woodwind_track229874__4_bassoon | Pitch Weight=1, Timing Weight=0, ins_cost=5, del_cost=5 | Tested

,tp,fp,fn,Clean False Positives,Mistake Event F1,Pitch Weight,Timing Weight,Insertion intercept,Deletion intercept,Rho (semitones),Pitch:Timing
0,1018,269,51,75,0.864177,16.0,1.0,16.00,16.00,1.00,16:1
1,1019,272,50,75,0.863559,4.0,1.0,4.00,4.00,1.00,4:1
2,1019,272,50,75,0.863559,3.0,1.0,3.00,3.00,1.00,3:1
3,1018,273,51,75,0.862712,2.0,1.0,2.00,2.00,1.00,2:1
4,1017,272,52,75,0.862595,6.0,1.0,6.00,6.00,1.00,6:1
5,1016,271,53,75,0.862479,12.0,1.0,12.00,12.00,1.00,12:1
6,1018,275,51,75,0.861981,4.0,1.0,5.00,5.00,1.25,4:1
7,1017,274,52,75,0.861864,16.0,1.0,20.00,20.00,1.25,16:1
8,1016,273,53,75,0.861747,8.0,1.0,8.00,8.00,1.00,8:1
9,1018,277,51,75,0.861252,3.0,1.0,3.75,3.75,1.25,3:1


Reports: /Users/sarah/Desktop/shasha/Attune/benchmarks/results/sweeps/balanced_212_v3/mistake_ratio_rho/4c6ea891f4eb104d
Winner: {'artifacts': {'test__brass_track222037__4_tuba': {'path': '/Users/sarah/Desktop/shasha/Attune/benchmarks/results/sweeps/balanced_212_v3/mistake_ratio_rho/4c6ea891f4eb104d/predictions/test__brass_track222037__4_tuba/c3b30e6cd8752118.json', 'sha256': '00553584ce27d44bfa8744ab7babf4f039d7d2aa67614b3933e8ae52928569b2'}, 'test__brass_track222070__3_trombone': {'path': '/Users/sarah/Desktop/shasha/Attune/benchmarks/results/sweeps/balanced_212_v3/mistake_ratio_rho/4c6ea891f4eb104d/predictions/test__brass_track222070__3_trombone/c3b30e6cd8752118.json', 'sha256': '9c66b1e5b079759ee1f741becf4a3c1a40477aec3a3865e4ec4852a3dee47c36'}, 'test__brass_track222182__1_trumpet': {'path': '/Users/sarah/Desktop/shasha/Attune/benchmarks/results/sweeps/balanced_212_v3/mistake_ratio_rho/4c6ea891f4eb104d/predictions/test__brass_track222182__1_trumpet/c3b30e6cd8752118.json', 'sha256':

## Provenance and resumption

The runner verifies the currently selected pitch/note winners, exact Coco-E source contract, and hashes of all 585 merged-note caches before scheduling alignment work. Initial notes are reused, but previous fixed-gap predictions are not. Per-candidate checkpoints resume interrupted runs. The result tables identify each ratio, rho and effective gap intercept; `best.json` records the experimental model. Historical reports and app defaults remain unchanged.


The previous linked Coco-E frontend must be complete. This experiment does not launch or refresh upstream sweeps.
